# Workshop 3: Sequential Movie Recommendation with a Dueling Double DQN on MovieLens 32M

**Course:** Machine Learning (2026-III), Computer Engineering, Universidad Distrital Francisco José de Caldas  
**Professor:** Eng. Carlos Andrés Sierra, M.Sc.  
**Team 7:** Andrés Felipe Ruiz Vasallo, Josep Emanuel León Joya, Matthew Stev Toro Mondragón

This notebook is a read-only walkthrough of Workshop 3. It does not train anything: every table is loaded from the JSON files written by the pipeline (`data_prep.py`, `env.py`, `train.py`, `evaluate.py`), so the numbers shown here are the same ones reported in the README and in `group7_workshop3.pdf`.

1. Leak-free data protocol and RL sessions built from real ratings recorded after the Workshop 1/2 training cut.
2. The `Gymnasium` environment `MovieLensRecEnv` and its random-policy validation.
3. Hyperparameter and architecture search for the Dueling Double DQN agent.
4. Held-out test comparison against Workshop 1 and Workshop 2 baselines, with paired statistics.
5. Figures (training curves, exploration, test metrics, trajectories, sample efficiency).

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, display

# Locate the repository root (the folder that contains workshop3/) from any working directory
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "workshop3" / "src" / "env.py").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from workshop3.src.baselines import GreedyFunkSVDPolicy
from workshop3.src.env import MovieLensRecEnv

W3_DIR = REPO_ROOT / "workshop3"


def load_json(relative_path: str) -> dict:
    with open(W3_DIR / relative_path, "r", encoding="utf-8") as f:
        return json.load(f)


metrics = load_json("report/metrics_summary.json")
dataset_stats = load_json("data/dataset_stats.json")
random_diag = load_json("report/random_diagnostics.json")
print(f"Repository root: {REPO_ROOT}")
print(f"Test sessions evaluated: {metrics['num_test_users']}")

## 1. Leak-free data protocol

Workshop 3 reuses the Workshop 1 sample (15,000 users, seed 42, iterative 10-core filter) and the global chronological 80/10/10 split shared by Workshops 1 and 2. FunkSVD (k = 20) is trained on the training split only. An RL session is the chronological list of a warm user's real ratings inside one later period: RL train and validation sessions come from the validation period, and RL test sessions come from the test period, the same held-out future that Workshops 1 and 2 report on. The candidate slate of a session holds its first 50 ratings; shorter sessions are padded with masked slots. A rating of 4.0 or more counts as a like.

In [ ]:
keys = [
    "total_raw_ratings", "total_raw_users", "filtered_ratings", "filtered_users", "filtered_movies",
    "train_ratings_global", "clean_val_ratings", "clean_test_ratings", "train_end_utc", "val_end_utc",
    "funksvd_test_rmse", "rl_train_users", "rl_val_users", "rl_test_users", "rl_train_test_user_overlap",
    "mean_valid_slots_test", "mean_liked_slots_test", "like_ratio_train_sessions", "like_ratio_test_sessions",
]
display(pd.DataFrame({"value": [dataset_stats[k] for k in keys]}, index=keys))

## 2. The Gymnasium environment

`MovieLensRecEnv` exposes `Discrete(50)` actions (candidate slots) and a 250-dimensional observation: the cosine similarity between each candidate's FunkSVD factor and the session vector, the static FunkSVD score, novelty, the unavailable-slot mask, the user factor, the session vector, and ten progress scalars. The session vector starts at the user's FunkSVD factor and moves toward liked items and away from missed ones. A hit (rating >= 4.0) earns a relevance reward plus novelty and genre-diversity bonuses; a miss costs a small penalty, a repeated or padding slot costs more, and the user abandons the session after five consecutive misses. The cell below replays one test session with the greedy FunkSVD policy (Workshop 1).

In [ ]:
env = MovieLensRecEnv(split="test", horizon=10, seed=42)
print(f"Observation space: {env.observation_space.shape} | Action space: Discrete({env.action_space.n})")

policy = GreedyFunkSVDPolicy()
obs, info = env.reset(seed=42, options={"user_idx": int(env.split_user_indices[0])})
done = False
while not done:
    action = policy.select_action(obs, info)
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
print(env.render())

### Random-policy validation (100 episodes)

A uniform random policy over all 50 slots checks reward bounds, early termination, and that every action is reachable (`unique_actions_sampled` must equal 50).

In [ ]:
display(pd.DataFrame({"value": random_diag}).rename_axis("diagnostic"))

## 3. Hyperparameter and architecture search

Eight one-at-a-time configurations (learning rate, discount factor, epsilon decay, L2 weight decay) are trained for two Q-network architectures: a flat dueling MLP over the whole state and a slot-shared dueling network that scores every slot with the same MLP. The run with the highest smoothed validation return is retrained as the champion, and a gamma = 0 copy of it is trained as the myopic (contextual bandit) ablation.

In [ ]:
hp = pd.DataFrame(metrics["hyperparameter_table"])
cols = ["config_id", "architecture", "lr", "gamma", "epsilon_decay", "weight_decay", "episodes",
        "smoothed_val_return", "best_val_return", "val_ndcg_at_10"]
display(hp[[c for c in cols if c in hp.columns]].sort_values("smoothed_val_return", ascending=False))

## 4. Held-out test comparison

Every policy is evaluated once on every RL test session, at K = 10 and K = 20. Greedy FunkSVD is the Workshop 1 model and Greedy HybridNeuMF the official Workshop 2 model, both acting greedily inside the same environment.

In [ ]:
table_cols = ["mean_return", "std_return", "precision_at_k", "recall_at_k", "ndcg_at_k", "map_at_k", "hr_at_k",
              "catalog_coverage", "mean_novelty_bits", "intra_list_diversity"]
for horizon in ("test_k10", "test_k20"):
    print(horizon)
    display(pd.DataFrame(metrics[horizon]).T[table_cols].sort_values("mean_return", ascending=False))

In [ ]:
rows = []
for pair, block in metrics["paired_comparisons"].items():
    if not isinstance(block, dict) or "policy_a" not in block:
        continue
    for metric_name in ("return", "ndcg_at_10"):
        s = block[metric_name]
        rows.append({"pair (A - B)": pair, "metric": metric_name, "mean_diff": s["mean_diff"],
                     "ci95": f"[{s['ci95_low']}, {s['ci95_high']}]", "wilcoxon_p": s["wilcoxon_p_value"],
                     "A better / B better / tied": f"{s['a_better_users']} / {s['b_better_users']} / {s['tied_users']}"})
display(pd.DataFrame(rows))

### Workshop 1 and Workshop 2 references

The static Workshop 1 and Workshop 2 numbers come from their own reports and summary files and use a different protocol (sampled-negative top-10 ranking), so they are context, not a one-to-one comparison with the sequential sessions above.

In [ ]:
w1 = metrics["workshop1_comparison"]
w2 = metrics["workshop2_comparison"]
display(pd.DataFrame({k: v for k, v in w1.items() if isinstance(v, dict)}).T)
static_w2 = {k: v for k, v in w2["w2_hybrid_neumf_static"].items() if isinstance(v, (int, float))}
display(pd.DataFrame({"Workshop 2 HybridNeuMF (static)": static_w2}))

## 5. Figures (300 DPI)

Figure 2 is written by `python workshop3/src/env.py`; Figures 3 to 7 by `python workshop3/src/evaluate.py`.

In [ ]:
fig_dir = W3_DIR / "report" / "figures"
for fig_path in sorted(fig_dir.glob("fig*.png")):
    print(f"=== {fig_path.name} ===")
    display(Image(filename=str(fig_path), width=850))